# IOAI — 2025 Residential Camp Nlp Crosslingual Affect (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
!git clone -q https://github.com/duyngtr16061999/Cross_lingual_sentiment_classification clx
import json, pandas as pd
jl = lambda p: [json.loads(l) for l in open(p).read().splitlines() if l.strip()]
tr, va, dt = jl('clx/DE_train.json'), jl('clx/DE_valid.json'), jl('clx/DT_test_unlabeled.json')
pd.DataFrame({'text':[r['text'] for r in tr],'label':[int(r['label']) for r in tr]}).to_csv('train.csv', index=False)
pd.DataFrame({'id':range(len(va)),'text':[r['text'] for r in va]}).to_csv('test.csv', index=False)
pd.DataFrame({'id':range(len(dt)),'text':[r['text'] for r in dt]}).to_csv('target_unlabeled.csv', index=False)
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Cross-Lingual Affect Classification

> **Singapore IOAI 2025 — Residential Camp (NLP)**

Train on **German** labelled tweets and classify affect (**0/1**). The true target-language test (`target_unlabeled.csv`) has no public labels, so the **Submit** button grades on the held-out **source-language validation** set (`test.csv`); score = **accuracy**.

> The TF-IDF baseline is monolingual — real cross-lingual transfer needs a multilingual encoder (e.g. XLM-R); that is the intended improvement direction.

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

train = pd.read_csv('train.csv')          # text,label
test = pd.read_csv('test.csv')            # id,text
train['text'] = train['text'].astype(str)
test['text'] = test['text'].astype(str)
train.shape, test.shape, train['label'].value_counts().to_dict()

## Self-check on a held-out split (TF-IDF + Logistic Regression baseline)

In [ ]:
Xtr, Xva, ytr, yva = train_test_split(train['text'], train['label'],
                                      test_size=0.2, random_state=42, stratify=train['label'])
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=50000, sublinear_tf=True)
clf = LogisticRegression(max_iter=1000, C=4.0)
clf.fit(vec.fit_transform(Xtr), ytr)
pva = clf.predict(vec.transform(Xva))
print(f'val accuracy {accuracy_score(yva, pva):.4f}  macro-F1 {f1_score(yva, pva, average="macro"):.4f}')

## Train on all data, predict the test set, write `submission.csv` (`id,label`)

In [ ]:
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=50000, sublinear_tf=True)
clf = LogisticRegression(max_iter=1000, C=4.0)
clf.fit(vec.fit_transform(train['text']), train['label'])
pred = clf.predict(vec.transform(test['text']))
pd.DataFrame({'id': test['id'], 'label': pred}).to_csv('submission.csv', index=False)
print('wrote submission.csv', len(test))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)